# Instancia Premier League 2022-2023

Descarga desde Football-Data la base de esta temporada y la usa como conjunto de equipos para resolver un calendario doble Round-Robin nuevo con el modelo compartido `modelo_calendario.py`.

## Qué representa esta instancia

El CSV histórico de 2022-2023 se usa para obtener y validar los 20 clubes de esa temporada. No se busca reproducir el calendario oficial: el modelo genera una programación nueva de 38 jornadas y 380 partidos, con cada pareja jugando dos veces, una en cada estadio.

## Flujo del notebook

1. Localiza el proyecto e importa `resolver_instancia` desde `modelo_calendario.py`.
2. Descarga los partidos históricos desde Football-Data y toma de ellos la lista de equipos.
3. Resuelve la instancia dos veces: con parámetros predeterminados de Gurobi y con cortes y heurísticas desactivados.
4. Compara estado, objetivo, cota, gap, tiempo y nodos; luego muestra el calendario nuevo y una muestra de los datos de origen.

Las listas de clásicos, estadios compartidos, equipos europeos y jornadas críticas están vacías; esas restricciones no aportan efecto hasta completar los datos. El término `viajes` es una aproximación basada en jugar de visita en una jornada crítica, no una distancia real.

## Licencia para ejecutar

Antes de ejecutar la celda Python que resuelve el modelo, activa la licencia desde PowerShell o la terminal de Windows con el comando que entrega Gurobi: `grbgetkey CLAVE_ENTREGADA_POR_GUROBI` (reemplaza el texto por la clave real, sin escribirla en el notebook). Después reinicia el kernel y ejecuta las celdas. Si `grbgetkey` no se reconoce, instala Gurobi Optimizer para Windows o agrega su carpeta `bin` al `PATH`. La lectura del CSV por sí sola no requiere licencia.

In [ ]:
# Evaluar la instancia 2022-2023 con GurobiPy en dos configuraciones.
from pathlib import Path
import sys

import pandas as pd
import time
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))
from modelo_calendario import resolver_instancia

TEMPORADA = "2022-2023"
URL_CSV = str((Path.cwd().resolve().parent / "BASES" / "2022-2023.csv"))
PARAMETROS_MODELO = {
    "clasicos": [],
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "numero_equipos_esperado": 20,
}

configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}

TIEMPO_TOTAL = 3600
MARGEN_SEGUNDOS = 30
inicio = time.monotonic()
resultados = {}

for indice, (nombre, desactivar_cortes_heuristicas) in enumerate(configuraciones.items()):
    pendientes = len(configuraciones) - indice
    disponible = TIEMPO_TOTAL - (time.monotonic() - inicio) - MARGEN_SEGUNDOS

    if disponible < 1:
        print(f"Se omite {nombre}: se agotó el presupuesto de tiempo.")
        break

    tiempo_limite = max(1, int(disponible / pendientes))
    print(f"{nombre}: límite de Gurobi de {tiempo_limite} segundos.")

    resultados[nombre] = resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        tiempo_limite=tiempo_limite,
        **PARAMETROS_MODELO,
    )

reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
reporte["Objetivo (ObjVal)"] = reporte["Objetivo (ObjVal)"].fillna("No factible")
reporte["Gap (MIPGap)"] = reporte["Gap (MIPGap)"].fillna("No factible")
estado_map = {
    1: "OPTIMAL",
    2: "INFEASIBLE",
    3: "INF_OR_UNBD",
    4: "CUTOFF",
    5: "LOADED",
    6: "UNKNOWN",
}
reporte["Estado Gurobi"] = reporte["Estado Gurobi"].map(lambda v: estado_map.get(v, v))
display(reporte)

resultado_normal = resultados["Normal (Gurobi por defecto)"]
print(f"Partidos históricos descargados: {len(resultado_normal['partidos_historicos'])}")
print(f"Breaks: {resultado_normal['breaks']} | Viajes críticos: {resultado_normal['viajes_criticos']}")
display(resultado_normal["calendario"])
display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))